---
## Verification Plan

Run these checks in order to confirm the pipeline works:

### 1. Data Loading
```bash
python -c "
from vep_nachr2.data.loader import load_mutation_data
df = load_mutation_data()
assert df.shape[0] == 797, f'Expected 797, got {df.shape[0]}'
assert set(df['effect'].unique()) == {'LOF', 'GOF', 'No net effect'}
assert len(df['subunit'].unique()) == 16
print('PASS: Data loading')
"
```

### 2. Reference Sequences
```bash
python -c "
from vep_nachr2.data.reference import load_all_reference_sequences
seqs = load_all_reference_sequences('human')
assert len(seqs) == 16, f'Expected 16, got {len(seqs)}'
for gene, seq in seqs.items():
    assert len(seq) > 300, f'{gene} too short: {len(seq)}'
print('PASS: Reference sequences')
"
```

### 3. Feature Extraction (without PDB)
```bash
python -c "
from vep_nachr2.data.loader import load_mutation_data
from vep_nachr2.data.reference import load_all_reference_sequences
from vep_nachr2.features.orchestrator import FeatureOrchestrator
import numpy as np

df = load_mutation_data()
ref_seqs = load_all_reference_sequences('human')
orch = FeatureOrchestrator(verbose=False)
X, names = orch.extract(df, ref_seqs=ref_seqs, use_cache=False)

assert X.shape == (797, 66), f'Bad shape: {X.shape}'
assert np.isnan(X).sum() == 0, f'NaNs: {np.isnan(X).sum()}'
print(f'PASS: Features {X.shape}, 0 NaN')
"
```

### 4. PDB Availability (after downloading)
```bash
python -c "
from vep_nachr2.features.structural import get_pdb_availability
import pprint
avail = get_pdb_availability()
pprint.pprint(avail)
available = sum(1 for v in avail.values() if v)
print(f'PDBs available: {available}/{len(avail)}')
"
# Expected: at least 4/6 initially (6UW8, 7EKT, 6CNJ, 6PV7 have .cif files)
```

### 5. Model Training (smoke test)
```bash
python -c "
from vep_nachr2.data.loader import load_mutation_data
from vep_nachr2.data.reference import load_all_reference_sequences
from vep_nachr2.features.orchestrator import FeatureOrchestrator
from vep_nachr2.models.registry import build_model
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
from vep_nachr2.data.loader import make_subunit_group_key
import numpy as np

df = load_mutation_data()
y = df['effect'].map({'LOF': 0, 'No net effect': 1, 'GOF': 2}).values
ref_seqs = load_all_reference_sequences('human')
X, _ = FeatureOrchestrator(verbose=False).extract(df, ref_seqs=ref_seqs, use_cache=True)

groups = make_subunit_group_key(df)
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(cv.split(X, y, groups))

model = build_model('random_forest', {'n_estimators': 100, 'max_depth': 5})
model.fit(X[train_idx], y[train_idx])
y_pred = model.predict(X[test_idx])

f1 = f1_score(y[test_idx], y_pred, average='macro')
assert f1 > 0.30, f'Macro F1 too low: {f1:.4f}'
print(f'PASS: Baseline macro F1 = {f1:.4f}')
"
```

### 6. Full Experiment (complete test)
```bash
python scripts/run_experiment.py --test
# Expected: completes in 2-5 minutes with macro F1 ~0.45-0.55
```

### Verification Checklist

- [ ] Data: 797 variants, 16 genes, 3 classes
- [ ] Features: (797, 66) matrix, 0 NaN
- [ ] PDBs: ≥4 structures with .cif files
- [ ] DSSP: ≥4 structures with .dssp files (or acceptable: fill values used)
- [ ] Baseline: macro F1 > 0.30 (untuned RF)
- [ ] CV: nested cross-validation runs without errors
- [ ] All 10 models: buildable and trainable
- [ ] Ablation: feature-group dropping works
- [ ] Species transfer: 3 conditions evaluate correctly

---
## Implementation Phases

### Phase 1: Scaffold & Data ✅ COMPLETE
1. Created `VEP Nachr2/` directory structure
2. Wrote `pyproject.toml` and `environment.yml`
3. Copied `merging_data/final.xlsx` → `data/raw/final.xlsx`
4. Implemented `vep_nachr2/config.py` — all constants, PDB mapping, gene lists
5. Implemented `vep_nachr2/data/loader.py` — load, clean, label-encode (841→797)
6. Implemented `vep_nachr2/data/reference.py` — 16 human FASTA sequences
7. Wrote `scripts/download_pdbs.py` + `scripts/download_alphafold.py`

### Phase 2: Core Features ✅ COMPLETE
8. Implemented `vep_nachr2/features/base.py` — FeatureExtractor ABC
9. Implemented `PhysicochemicalExtractor` — 24 AAIndex features
10. Implemented `SubstitutionExtractor` — BLOSUM62 + Grantham
11. Implemented `PositionalExtractor` — position + subunit + species one-hot
12. Implemented `StructuralExtractor` — RSA, Cβ-density, B-factor, DSSP, is_unmappable
13. Implemented `FeatureOrchestrator` with caching

### Phase 3: Models & Training ✅ COMPLETE
14. Implemented `vep_nachr2/models/registry.py` — 10 models + HP spaces
15. Implemented `vep_nachr2/models/imbalance.py` — per-model strategy dispatch
16. Implemented `vep_nachr2/training/cross_validation.py` — nested CV + Optuna + species transfer
17. Implemented `vep_nachr2/training/evaluation.py` — macro F1, MCC, confusion matrix
18. Implemented `vep_nachr2/training/runner.py` — CLI + high-level orchestrators

### Phase 4: nAChR-Specific Features ✅ COMPLETE
19. Implemented `StructuralNachrExtractor` — TMD, ligand, interface features
20. Implemented `ConformationalExtractor` — α7 open/closed delta (7EKT vs 7EKO)
21. Pipeline verified end-to-end (797 × 66 → macro F1 0.46 baseline)

### Phase 5: Experiments 🔲 TO DO
22. Run species transfer CV (`python scripts/run_experiment.py --species-transfer`)
23. Run ablation study (`python scripts/run_experiment.py --ablation`)
24. Run full model comparison (`python scripts/run_experiment.py --compare`)
25. Analyze results in notebooks

### Phase 6: Future Extensions 🔲 PLANNED
- Download ESM-2 model, implement EmbeddingExtractor
- Add GNN on 3D structure graph (PyTorch Geometric)
- Download mouse/rat ortholog reference sequences
- Add conformational features for subunits beyond α7
- Support indels/frameshifts (separate model or binary flag)

---
## Data Flow — End to End

```
merging_data/final.xlsx  (unique_variants sheet, 841 rows)
    │
    ▼
vep_nachr2/data/loader.py::load_mutation_data()
    │  ┌──────────────────────────────────────────────┐
    │  │ 1. Standardize columns (COLUMN_MAPPING)       │
    │  │ 2. Drop 14 LOF/GOF ambiguous labels           │
    │  │ 3. Filter to substitutions only (drop 30)     │
    │  │ 4. Validate AA codes, positions               │
    │  │ 5. Standardize gene names (CHRNA1-CHRNG)      │
    │  │ 6. Standardize species (human/mouse/rat)      │
    │  └──────────────────────────────────────────────┘
    │  797 variants, 16 genes, 3 species, 3 classes
    ▼
vep_nachr2/data/reference.py::load_all_reference_sequences()
    │  ┌──────────────────────────────────────────────┐
    │  │ Load 16 FASTA files from                      │
    │  │ data/raw/reference_sequences/human/<GENE>.fasta│
    │  │ Returns: {gene: sequence_str}                 │
    │  └──────────────────────────────────────────────┘
    ▼
vep_nachr2/features/orchestrator.py::FeatureOrchestrator.extract()
    │
    ├── PhysicochemicalExtractor  (24 features)
    │   └── aaindex1 lookup → 8 props × {wt, mt, diff}
    │
    ├── SubstitutionExtractor  (3 features)
    │   └── BLOSUM62 + Grantham distance
    │
    ├── PositionalExtractor  (20 features)
    │   └── pos/gene_len + 16 subunit OH + 3 species OH
    │
    ├── StructuralExtractor  (7 features)  [requires PDB]
    │   └── PDB alignment → DSSP(RSA, SS) + KDTree(Cβ) + B-factor
    │       Fallback: fill values if PDB missing
    │
    ├── StructuralNachrExtractor  (7 features)  [requires PDB]
    │   └── TM annotations + pore axis + binding site + interface KDTree
    │
    ├── ConformationalExtractor  (5 features)  [α7 only, requires PDB]
    │   └── 7EKT vs 7EKO delta features
    │
    └── EmbeddingExtractor  (0 features)  [placeholder]
    │
    ▼
Feature matrix: (797, 66) + cache to data/processed/feature_cache_<hash>.pkl
    │
    ▼
vep_nachr2/training/cross_validation.py::nested_cross_validation()
    │
    │  ┌────────────────── OUTER CV ──────────────────┐
    │  │ 5-fold StratifiedGroupKFold (groups=subunit)  │
    │  │                                                │
    │  │  For each fold:                                │
    │  │  ┌──────────── INNER CV ────────────┐         │
    │  │  │ 50 Optuna trials (TPE sampler)    │         │
    │  │  │ 5-fold StratifiedKFold            │         │
    │  │  │ Objective: macro F1               │         │
    │  │  │ → Best hyperparameters found      │         │
    │  │  └──────────────────────────────────┘         │
    │  │                                                │
    │  │  Best model → fit on outer train               │
    │  │  → predict outer test → macro F1, MCC, etc.   │
    │  └────────────────────────────────────────────────┘
    │
    ▼
Results: JSON + CSV → results/baseline/
    Per-fold: metrics, best_params, test_genes
    Aggregate: mean ± std across 5 folds × 5 seeds
```

---
## Key Design Decisions (from planning session)

These 16 decisions were made after grilling through the VEP-ENAC architecture and dataset:

| # | Decision | Choice | Rationale |
|---|----------|--------|-----------|
| 1 | Classification target | 3-class: GOF/LOF/NNE | VEP-ENAC precedent; 14 LOF/GOF ambiguous dropped |
| 2 | Modification types | Substitutions only | BLOSUM/Grantham/structural features only make sense for point mutations |
| 3 | Species strategy | Human-primary + species transfer CV | Tests cross-species augmentation hypothesis |
| 4 | PDB structures | Per-receptor-type PDBs (5 exp + 2 AF) | One structure can't cover 16 diverse genes |
| 5 | Feature scope | VEP-ENAC clone + nAChR-specific extensions | Structural features were most important in ENaC ablation |
| 6 | Conformational features | Architecture-ready, α7 only initially | Only α7 has both open+closed structures (7EKT/7EKO) |
| 7 | Models | Classical ML (10) first, ESM-2/GNN ready | Classical baseline needed before deep learning |
| 8 | CV grouping | Gene-level LOSO primary + homology transfer secondary | Strictest generalization; no subunit leakage |
| 9 | Feature architecture | Pipeline with FeatureExtractor ABC | Each extractor independently testable + droppable for ablation |
| 10 | Class imbalance | Per-model strategy dispatch (v3 pattern) | Different models need different imbalance handling |
| 11 | Packaging | pyproject.toml + pip install -e . | Professional, reproducible, no conda required |
| 12 | Metrics | Macro F1 (Optuna objective) + MCC (reported) | Both handle class imbalance; F1 is standard, MCC is honest |
| 13 | Reference sequences | Human-only now, rodent-ready | Rodent variants mapped through human reference for structural features |
| 14 | Old VEP Nachr | Salvage notebooks + PDB mapping; ignore code | week1.ipynb and features.ipynb have useful analysis |
| 15 | Implementation priority | Full architecture first, then iterate | Clean foundation prevents technical debt |
| 16 | Dependencies | pip (pyproject.toml) + optional conda | Works with standard pip; conda only needed for mkdssp |

# VEP-nAChR2 — Project Reference Notebook

## What This Project Is

**Variant Effect Predictor for Nicotinic Acetylcholine Receptors (nAChR)**.

Predicts the **functional direction** of missense variants:
- **LOF** (Loss-of-Function) → label 0
- **No net effect** (NNE) → label 1
- **GOF** (Gain-of-Function) → label 2

This is NOT a generic pathogenicity predictor. It predicts *direction of effect* —
a harder, less-explored task. Most VEP tools (AlphaMissense, PolyPhen, CADD) predict
damaging vs benign but cannot distinguish GOF from LOF.

## Architecture

Modeled after VEP-ENAC (Noah Plingen, 2024) but built from scratch.
Covers 16 human nAChR genes with multi-species augmentation (human + mouse + rat).

---
## Quick Start

### 1. Install
```bash
cd "VEP Nachr2"
pip install -e .
# or: conda env create -f environment.yml
```

### 2. Test data loading
```python
from vep_nachr2.data.loader import load_mutation_data
df = load_mutation_data()  # 797 variants
```

### 3. Quick experiment
```bash
python scripts/run_experiment.py --test
```

### 4. Full experiment
```bash
python scripts/run_experiment.py --full
```

---
## Project Structure

```
VEP Nachr2/
├── pyproject.toml              # Package metadata, dependencies
├── environment.yml             # Conda environment
├── vep_nachr2/                 # Main package (pip install -e .)
│   ├── config.py               # ALL constants, PDB mapping, genes, paths
│   ├── data/
│   │   ├── loader.py           # Load final.xlsx, standardize, filter
│   │   └── reference.py        # Load reference sequences (FASTA)
│   ├── features/
│   │   ├── base.py             # FeatureExtractor ABC
│   │   ├── physicochemical.py  # 24 AAIndex features (8 props × wt/mt/diff)
│   │   ├── substitution.py     # 3 features (BLOSUM + Grantham)
│   │   ├── positional.py       # 20 features (pos + 16 gene OH + 3 species OH)
│   │   ├── structural.py       # 7 core PDB features (RSA, Cβ, B-factor, DSSP)
│   │   ├── structural_nachr.py # 7 nAChR-specific (TMD, ligand, interface)
│   │   ├── conformational.py   # 5 open/closed delta (α7 only)
│   │   ├── embeddings.py       # Placeholder for ESM-2
│   │   └── orchestrator.py     # Runs all extractors, caches results
│   ├── models/
│   │   ├── registry.py         # 10 models + HP search spaces
│   │   └── imbalance.py        # Per-model imbalance strategy dispatch
│   └── training/
│       ├── cross_validation.py # Nested CV + Optuna + species transfer
│       ├── evaluation.py       # Macro F1, MCC, confusion matrix, etc.
│       └── runner.py           # CLI + high-level orchestrators
├── scripts/
│   ├── run_experiment.py       # Main entry point
│   └── download_pdbs.py        # Download PDB + DSSP
├── notebooks/
├── data/
│   ├── raw/
│   │   ├── final.xlsx          # 842 unique variants (from merging_data/)
│   │   ├── structure_files/    # PDB .pdb/.cif/.dssp files
│   │   └── reference_sequences/human/  # 16 FASTA files
│   └── processed/              # Cached features
├── results/                    # Experiment outputs
└── tests/
```

---
## Data

### Source: `data/raw/final.xlsx`
Copied from `merging_data/final.xlsx` (the merged dataset of all 3 source files:
nachr_db_manual.xlsx + human_manual2.xlsx + mouse_data_manual.xlsx).

### Sheets
- **unique_variants** (841 rows) — deduplicated, one row per unique variant. USE THIS.
  Contains variants where the same mutation appears in multiple papers merged into one row.
- **all_variants** — all rows including duplicates across papers (more rows, redundant info)

### Data Reduction: 841 → 797

| Step | Removed | Reason |
|------|---------|--------|
| Load unique_variants | — | 841 rows |
| Drop ambiguous effects | 14 rows | Labeled "LOF/GOF" — variant has been reported as BOTH in literature (most are CHRNA7 positions 237, 254, 255, plus CHRNB2 and CHRND). These positions genuinely show context-dependent effects and can't be classified cleanly. |
| Drop non-substitutions | 30 rows | 15 Deletions, 10 Stop mutations, 4 Frameshifts, 1 Insertion. These can't use substitution-based features (BLOSUM, Grantham) and have NaN positions — keeping them would require a separate model. |
| **Final** | **797 rows** | Clean 3-class substitution dataset |

### Final Distribution
- **797 variants**: Human (542), Rat (174), Mouse (80), NaN species (1)
- **Effects**: LOF (393, 49.3%), GOF (211, 26.5%), No net effect (193, 24.2%)
- **16 genes**: All present, CHRNA7 is largest (268 in raw), CHRNA2 smallest (~10)
- **Class imbalance**: ~2:1:1 ratio (LOF dominant). Handled by per-model imbalance strategies.

### Label encoding
```python
LOF → 0, No net effect → 1, GOF → 2  # sklearn-compatible 3-class
```

---
## Genes & PDB Mapping

### 16 nAChR Genes

| Gene | Homology Class | PDB | Chain | Source |
|------|---------------|-----|-------|--------|
| CHRNA1 | alpha | 6UW8 | A | Experimental (3.20 Å) |
| CHRNA2 | alpha | 6CNJ | A | Experimental (homology: α4) |
| CHRNA3 | alpha | 6PV7 | A | Experimental (2.80 Å) |
| CHRNA4 | alpha | 6CNJ | A | Experimental (3.30 Å) |
| CHRNA5 | alpha | 6PV7 | A | Experimental (homology: α3) |
| CHRNA6 | alpha | 6CNJ | A | Experimental (homology: α4) |
| CHRNA7 | alpha | 7EKT | A | Experimental (3.20 Å) |
| CHRNA9 | alpha | AF-Q9UGM1 | A | AlphaFold |
| CHRNA10 | alpha | AF-Q13002 | A | AlphaFold |
| CHRNB1 | beta | 6UW8 | B | Experimental |
| CHRNB2 | beta | 6CNJ | B | Experimental |
| CHRNB3 | beta | 6PV7 | B | Experimental (homology: β4) |
| CHRNB4 | beta | 6PV7 | B | Experimental |
| CHRND | special | 6UW8 | D | Experimental |
| CHRNE | special | 6UW8 | E | Experimental |
| CHRNG | special | 6UW8 | E | Experimental (homology: ε) |

### Homology Classes (for cross-family transfer experiments)
- **alpha**: CHRNA1-7, A9, A10 (9 genes)
- **beta**: CHRNB1-4 (4 genes)
- **special**: CHRND, CHRNE, CHRNG — δ/ε/γ (3 genes)

---
## PDB Files to Download

### Step 1: Install DSSP (required for structural features)
```bash
conda install -c salilab dssp
# Or on Ubuntu: sudo apt install dssp
# Or on macOS: brew install brewsci/bio/dssp
```

### Step 2: Download experimental PDB structures
```bash
cd "VEP Nachr2"
python scripts/download_pdbs.py
```
This downloads from RCSB PDB (https://files.rcsb.org) and runs mkdssp automatically.

### Step 3: Download AlphaFold structures (CHRNA9 & CHRNA10)
```bash
python scripts/download_alphafold.py
```
Downloads AF-Q9UGM1 (CHRNA9) and AF-Q13002 (CHRNA10) from AlphaFold DB (https://alphafold.ebi.ac.uk).
AlphaFold structures are monomeric — features requiring pentameric context (interface_proximity,
subunit_burial) will use fallback values for these two genes.

### Required PDBs (5 experimental + 1 conformational + 2 AlphaFold)

| PDB ID | Structure | Resolution | Covers | Download |
|--------|-----------|------------|--------|----------|
| **6UW8** | Human muscle α1β1δε | 3.20 Å | CHRNA1, CHRNB1, CHRND, CHRNE, CHRNG | `python scripts/download_pdbs.py` |
| **7EKT** | Human α7 (closed/resting) | 3.20 Å | CHRNA7 | `python scripts/download_pdbs.py` |
| **7EKO** | Human α7 (open/activated) | 3.60 Å | CHRNA7 conformational delta | `python scripts/download_pdbs.py` |
| **6CNJ** | Human α4β2 neuronal | 3.30 Å | CHRNA4, CHRNB2, CHRNA2, CHRNA6 | `python scripts/download_pdbs.py` |
| **6PV7** | Human α3β4 neuronal | 2.80 Å | CHRNA3, CHRNB4, CHRNA5, CHRNB3 | `python scripts/download_pdbs.py` |
| **AF-Q9UGM1** | CHRNA9 AlphaFold | N/A | CHRNA9 | `python scripts/download_alphafold.py` |
| **AF-Q13002** | CHRNA10 AlphaFold | N/A | CHRNA10 | `python scripts/download_alphafold.py` |

### After downloading PDBs
Just re-run your experiment — the structural features will automatically use the
newly available PDB data. No code changes needed. The `FeatureOrchestrator` detects
PDB files on disk and switches from fill-values to real structural features.

```bash
# Verify PDB availability
python -c "
from vep_nachr2.features.structural import get_pdb_availability
import pprint
pprint.pprint(get_pdb_availability())
"

# Clear feature cache (so structural features are recomputed with PDB data)
rm data/processed/feature_cache_*.pkl

# Run with structural features
python scripts/run_experiment.py --test
```

### That's it — nothing else needed to run the full pipeline.

---
## Protein Sequences

### Source
The reference sequences come from the `protein_scequences/` folder at the repo root.
These were downloaded via NCBI Datasets for all 16 human nAChR subunits.

### How we use them
1. **Position validation**: Verify that each variant's `wildtype_aa` matches the
   reference sequence at the given position.
2. **Position normalization**: Each gene has a different sequence length — the
   `position_normalized` feature divides position by gene length.
3. **PDB alignment**: The reference sequence is aligned to the PDB chain using
   BioPython PairwiseAligner (BLOSUM62, global mode) to map UniProt positions
   to PDB residue IDs for structural feature computation.

### Location
- Human: `data/raw/reference_sequences/human/<GENE>.fasta` (16 files)
- Mouse/Rat: Not yet populated (future: `data/raw/reference_sequences/mouse/`, `rat/`)

The FASTA files were copied from `protein_scequences/<gene>/ncbi_dataset/data/protein.faa`
using the RefSeq canonical isoform for each gene.

---
## Features — Detailed Explanation

**Total: 66 features across 7 extractor groups**

Each feature is included based on known biophysics of ion channel function.
Amino acid substitutions affect protein function through changes in:
- **Local chemistry** (charge, polarity, size) → captured by physicochemical features
- **Evolutionary constraint** (is this change tolerated?) → captured by substitution features
- **3D location** (where in the structure?) → captured by positional + structural features
- **Channel-specific context** (pore, gate, binding site?) → captured by nAChR-specific features

---

### Group 1: Physicochemical Features (24 features)

**Why:** When one amino acid is swapped for another, the local chemical environment changes.
The magnitude and direction of this change determines whether the protein is disrupted.
Three views capture the full picture: what was there (wt), what replaced it (mt), and the delta (change).

**8 AAIndex Properties Used:**

| Property | AAIndex ID | Why It Matters for nAChR |
|----------|-----------|--------------------------|
| **Hydrophobicity** | EISD840101 | Controls membrane insertion and helix packing. nAChR has 4 TM helices per subunit — hydrophobic mismatch disrupts gating. Eisenberg consensus scale. |
| **Polarity** | GRAR740102 | Affects hydrogen bonding at subunit interfaces and ligand binding. nAChR agonist binding involves polar interactions with ACh/nicotine. Grantham scale. |
| **Volume** | KRIW790103 | Steric effects: replacing a small residue with a large one in the pore (M2 helix) physically blocks ion flow. Krigbaum & Komoriya scale. |
| **Molecular Weight** | FASG760101 | Proxy for side-chain size. Large changes (e.g., Gly→Trp) often cause misfolding or steric clashes. Fasman scale. |
| **Charge** | KLEP840101 | nAChR is a cation channel — charged residues in the pore, selectivity filter, and ring of charges in M2 directly affect conductance. Klein scale. |
| **Isoelectric Point** | ZIMJ680104 | Combined charge/hydrophobicity measure. Zimmerman scale. |
| **Aromaticity** | Binary (F/W/Y/H) | Aromatic residues are enriched in the agonist binding site (C-loop, "aromatic box"). Loss/gain of aromaticity affects ligand binding. |
| **SS Preference** | CHOP780201 | Helix vs sheet propensity. nAChR is mostly helical — variants that disrupt helix formation in TM domains cause misfolding. Chou & Fasman helix propensity. |

**Output:** For each property: `wt_{prop}` (8), `mt_{prop}` (8), `diff_{prop}` (8) = 24 features.
All normalized to [0, 1] for comparability across scales.

---

### Group 2: Substitution Features (3 features)

**Why:** Not all amino acid changes are equal. Evolution has encoded which substitutions
are "acceptable" (BLOSUM62) and biophysics tells us how dramatic a change is (Grantham).

| Feature | What It Measures | Why It Matters |
|---------|-----------------|----------------|
| **blosum_score** | Raw BLOSUM62 substitution score | Evolutionary constraint: negative = rarely observed, positive = common. GOF/LOF variants often have strongly negative BLOSUM scores (rare changes = disruptive). |
| **blosum_normalized** | Min-max normalized to [0, 1] | Same information, scaled for ML models that expect [0,1] range. |
| **grantham_distance** | Physicochemical distance (composition + polarity + volume) | Bigger distance = more dramatic chemical change. Grantham distances >100 indicate radical substitutions. Many pathogenic nAChR variants have high Grantham scores. |

**Why BLOSUM62 AND Grantham?** They capture different things: BLOSUM is evolutionary (what nature tolerates), Grantham is biophysical (how big the chemical change is). A substitution can be evolutionarily rare (low BLOSUM) but chemically conservative (low Grantham) — or vice versa.

---

### Group 3: Positional Features (20 features)

**Why:** *Where* a mutation occurs matters as much as *what* changes.

| Feature | Description | Why It Matters |
|---------|-------------|----------------|
| **position_normalized** | Position / gene sequence length | Relative position in the protein matters: N-terminal domain vs TM helix vs intracellular loop have different functional roles. Normalized so genes of different lengths are comparable. |
| **subunit_<GENE>** (16) | One-hot encoding of gene identity | Each gene has different baseline properties. CHRNA7 is homomeric, CHRNA1 is muscle-type — the model needs to know which context a variant is in. |
| **species_<X>** (3) | One-hot: human, mouse, rat | Same variant in different species may behave differently. Species-specific effects are captured here. |

---

### Group 4: Core Structural Features (7 features, requires PDB)

**Why:** PDB structures tell us about the 3D environment of each residue — is it buried
in the core, or exposed on the surface? Is it in a rigid or flexible region? These are
the features VEP-ENAC found most important in ablation studies.

| Feature | Source | What It Measures | Why It Matters for nAChR |
|---------|--------|-----------------|--------------------------|
| **rsa** | DSSP ASA / MAX_ASA | Relative Solvent Accessibility (0=buried, 1=exposed) | Buried core mutations disrupt folding. Exposed surface mutations alter interactions. Pore-lining residues have intermediate RSA. |
| **cbeta_density** | PDB KDTree (10Å radius) | Local atomic packing density | Well-packed regions (high density) tolerate fewer changes. Low-density regions (loops) are more permissive. |
| **b_factor** | PDB B-factor (residue avg) | Thermal mobility / flexibility | High B-factor = flexible (loops, termini). Low = rigid (helix core). Mutations in rigid regions are more destabilizing. |
| **dssp_helix** | DSSP → one-hot | 1 if α/3-10/π helix | nAChR TM domains are helical — helix-breaking mutations cause misfolding. |
| **dssp_sheet** | DSSP → one-hot | 1 if β-strand/bridge | nAChR extracellular domain has β-sheet-rich ligand-binding region. |
| **dssp_coil** | DSSP → one-hot | 1 if loop/turn/irregular | Coil = 1 minus (helix + sheet). One-hot sum always = 1. |
| **is_unmappable** | Alignment success | 1 if position not in PDB, 0 if mapped or pseudo-resolved | Flags positions we can't get structural data for (disordered termini, internal loops). Terminal IDR gaps are pseudo-resolved to 0. |

**PDB-to-variant mapping:** Each variant's UniProt position is aligned to the PDB chain using BioPython PairwiseAligner (BLOSUM62, global mode). Unmappable positions (outside ATOM range, loops not resolved) receive fill values appropriate for disordered regions (rsa=1.0, coil, chain_max B-factor).

---

### Group 5: nAChR-Specific Structural Features (7 features, requires PDB)

**Why:** nAChR has unique structural features not captured by generic structural features.
These features encode nAChR-specific biophysics that directly relate to GOF/LOF mechanism.

| Feature | Description | Why It Matters for GOF/LOF |
|---------|-------------|---------------------------|
| **tm_helix** | 0-4: which TM helix (TM1-TM4), 0=non-TM | TM2 (M2) lines the pore — mutations here directly affect ion conductance. TM4 faces lipids — mutations affect assembly. Each helix has different functional roles. |
| **tm_depth** | Position within TM helix, 0=extracellular edge, 1=intracellular | The "hydrophobic gate" at the intracellular end of M2 controls channel opening. Depth tells us if a mutation is near the gate. |
| **pore_distance** | Distance from residue CA to pore axis (avg of M2 helix centers) | Pore-lining mutations affect conductance and selectivity. Distant mutations affect gating or assembly. Distance to pore is a key determinant of effect type. |
| **ligand_proximity** | Distance to C-loop + orthosteric pocket (loops A-F) | Agonist binding site mutations directly affect activation. Mutations in the C-loop "aromatic box" (the ACh binding pocket) are strong GOF candidates. |
| **interface_proximity** | Minimum distance to nearest neighbor chain CA | Subunit interfaces are where gating conformational changes propagate. Interface mutations affect cooperativity, assembly, and the ECD-TMD coupling that transduces binding → opening. |
| **interface_contacts** | Count of neighbor chain atoms within 5 Å | Dense interface contacts = more constrained = mutations more disruptive. |
| **subunit_burial** | Fraction of SASA buried by pentamer assembly | Identifies assembly interfaces vs exposed surfaces. Buried interface residues are enriched for disease-causing variants. |

**TM annotations** are from UniProt/OPM for human subunits. For homology-mapped genes (CHRNA2→α4, CHRNA5→α3, CHRNG→ε), the parent subunit's TM boundaries are used.

---

### Group 6: Conformational Features (5 features, α7 only, requires PDB)

**Why:** GOF/LOF is inherently about conformational change — the channel transitions
between resting/closed and activated/open states. Features that change between states
are mechanistically informative.

Active only for CHRNA7 (7EKT closed vs 7EKO open, both solved experimentally).
Returns zeros for all other genes (architecture ready for future structures).

| Feature | Description | Why It Matters |
|---------|-------------|----------------|
| **delta_rsa** | RSA(open) - RSA(closed) | Residues that become more exposed on opening (e.g., extracellular side of M2) — mutations here may affect the opening transition. |
| **delta_bfactor** | B-factor(open) - B-factor(closed) | Regions that become more rigid/flexible during gating. Increased rigidity on opening suggests involvement in the activated state. |
| **delta_interface** | Interface distance(open) - interface distance(closed) | Interface rearrangements are central to gating. The ECD-TMD interface rotates during activation. |
| **ca_rmsd** | Cα displacement between open/closed states | Direct measure of conformational change magnitude. Large RMSD = key gating regions (C-loop, M2-M3 linker). |
| **pore_radius_change** | M2 pore radius(open) - M2 pore radius(closed) | The pore opening itself. M2 helices tilt outward on activation — mutations that affect this tilt cause GOF/LOF. |

---

### Group 7: Embedding Features (0 features — placeholder)

Architecture-ready placeholder for ESM-2 protein language model embeddings.
When implemented, will provide per-position embedding vectors (1280-dim for esm2_t33)
capturing evolutionary and structural information learned from 250M protein sequences.

**Why not implemented yet:** Requires `facebook/esm` library and ~2GB model download.
Planned as a future experiment.

---

### Feature Summary

| Context | Without PDB | With PDB | With AF | Full |
|---------|------------|----------|---------|------|
| Features active | Groups 1-3, 7 | + Group 4 | + partial Group 5 | All 7 groups |
| Total features | 47 | 54 | 59-61 | 66 (α7=66, others=61) |
| PDB-requiring | 0 | 7+7 (core+nachr) | +5 (conformational, α7 only) | 19-24 |

---
## Models — Detailed Explanation

**10 machine learning models** spanning different algorithmic families.
This diversity is intentional: different model types capture different patterns in the data.

### Why 10 Models?

1. **No single model is best for small datasets.** With 797 samples and 66 features, different
   models make different bias-variance tradeoffs. Ensemble methods (RF, LightGBM, XGBoost)
   may excel with non-linear interactions, while regularized linear models (LR, SVM) may
   generalize better with limited data.

2. **Thesis structure.** Model comparison is a key experiment. Showing that simpler models
   perform comparably to complex ones is a finding in itself.

3. **VEP-ENAC precedent.** Their ablation showed LightGBM and CatBoost performed best on
   ion channel data. We replicate and extend their model lineup.

### Model Details

| # | Model | Type | Scaling? | Why Included |
|---|-------|------|----------|-------------|
| 1 | **Logistic Regression** | Linear, L1/L2 | Yes | Baseline. Interpretable coefficients via SHAP. ElasticNet penalty (saga solver) handles correlated features. |
| 2 | **SVM (RBF)** | Kernel | Yes | Captures non-linear decision boundaries with small data. RBF kernel handles complex feature interactions without overfitting. |
| 3 | **SVM (Linear)** | Linear | Yes | Linear baseline for SVM family. Useful to compare: if RBF >> Linear, the problem is non-linear. |
| 4 | **Random Forest** | Tree ensemble | No | Robust default. Handles mixed feature types well. Hundreds of trees average out noise from small dataset. |
| 5 | **LightGBM** | Gradient boosting | No | Typically best performer on tabular biological data. Leaf-wise growth with regularization. VEP-ENAC's top model. |
| 6 | **KNN** | Distance-based | Yes | Simple non-parametric baseline. With imputation pipeline for missing structural features. |
| 7 | **MLP** | Neural network | Yes | Shallow neural net (1 hidden layer, 2-32 units). Tests whether non-linear transformations beyond tree ensembles help. |
| 8 | **Gaussian NB** | Probabilistic | No | Fast baseline. Works surprisingly well with normalized features despite feature independence assumption. |
| 9 | **XGBoost** | Gradient boosting | No | Alternative to LightGBM. Different regularization approach. Good for comparison. |
| 10 | **CatBoost** | Gradient boosting | No | Handles categorical features natively. VEP-ENAC found it competitive with LightGBM for ion channel data. |

### Hyperparameter Optimization

**Optuna with TPE (Tree-structured Parzen Estimator) sampler:**
- **50 trials per outer fold** (30 for comparison, 10 for quick tests)
- **MedianPruner**: stops unpromising trials early
- **Objective: macro F1** (treats GOF, LOF, NNE equally regardless of class size)
- **HP spaces aggressively capped** for small inner folds (~51 samples minimum)

**Dynamic HP capping:** When inner folds are very small (e.g., 51 samples for a rare
gene in leave-one-out CV), certain HPs are automatically reduced:
- KNN `n_neighbors` capped to minority class count - 1
- MLP hidden unit count limited to n_train / 20
- Tree depths capped at 2-5 (prevent overfitting)

### Class Imbalance Handling

LOF is 49% of data. Simple accuracy would be misleading (predicting "LOF" always = 49% accuracy).
Each model type gets a strategy matched to its algorithm:

| Strategy | Models | Mechanism | Why This Model? |
|----------|--------|-----------|-----------------|
| **cost_sensitive** | LR, SVM, LightGBM, CatBoost | `class_weight='balanced'` | These models natively support per-sample or per-class weighting. Automatically up-weights minority class errors. |
| **brfc** | Random Forest | `BalancedRandomForestClassifier` (imblearn) | RF with balanced bootstrap sampling + balanced class weighting. More sophisticated than simple class_weight for bagging. |
| **ros** | KNN, MLP, GaussianNB | `RandomOverSampler` in imblearn Pipeline | These models don't support class_weight. Synthetic minority oversampling is applied BEFORE fitting. ROS chosen over SMOTE because with small data, synthetic samples can introduce noise. |
| **xgb_inverse** | XGBoost | `sample_weight = 1/class_frequency` | XGBoost's `scale_pos_weight` only works for binary. For multi-class, manual inverse-frequency sample weights are computed and passed to `fit()`. |

### Expected Performance

Based on VEP-ENAC results and our preliminary tests:
- **Baseline (no tuning):** macro F1 ≈ 0.45 (Random Forest, 1 fold)
- **With HP tuning:** macro F1 ≈ 0.55-0.65 expected
- **Best model prediction:** LightGBM or CatBoost (gradient boosting handles small tabular data best)
- **Structural features expected to add:** +0.05-0.10 macro F1 (VEP-ENAC's most important feature group)

---
## Cross-Validation Strategy — Detailed Explanation

### The Core Problem: Data Leakage

With 16 nAChR genes sharing ~30% sequence identity, a model can "cheat" by memorizing
gene-specific patterns instead of learning biophysical principles. If CHRNA7 position 250
appears in both train and test, the model just learns "α7:250 = GOF" rather than learning
that "a hydrophobic→charged change in the M2 pore region causes GOF."

**Our solution: nested cross-validation at the gene level.**

### Nested CV Architecture

```
┌─────────────────────────────────────────────────────────┐
│                    OUTER CV LOOP                         │
│  (5 folds, StratifiedGroupKFold, groups=subunit gene)    │
│                                                          │
│  For each outer fold:                                    │
│  ┌──────────────────────────────────────────────────┐   │
│  │  Train on 12-13 genes → Test on 3-4 held-out genes │   │
│  │                                                    │   │
│  │  ┌──────────────────────────────────────────┐    │   │
│  │  │         INNER CV LOOP                     │    │   │
│  │  │  (5 folds, StratifiedKFold)               │    │   │
│  │  │                                           │    │   │
│  │  │  For 50 Optuna trials:                    │    │   │
│  │  │    For each inner fold:                   │    │   │
│  │  │      Train on 80% of outer train          │    │   │
│  │  │      Validate on 20% of outer train       │    │   │
│  │  │    → Mean macro F1 across inner folds     │    │   │
│  │  │                                           │    │   │
│  │  │  Best HP set found → build best model     │    │   │
│  │  └──────────────────────────────────────────┘    │   │
│  │                                                    │   │
│  │  Fit on FULL outer train → predict outer test     │   │
│  │  → Report: macro F1, MCC, confusion matrix        │   │
│  └──────────────────────────────────────────────────┘   │
│                                                          │
│  Final: mean ± std across 5 folds × 5 seeds              │
└─────────────────────────────────────────────────────────┘
```

### Why Nested CV?

**Without nesting (flat CV + HP tuning on full data):** The HP search "sees" the test
data, choosing parameters that happen to work on those specific test folds. This inflates
reported performance — a form of information leakage called "optimism bias."

**With nesting:** HPs are tuned on inner training data only. The test fold is NEVER seen
during HP selection. The outer performance estimate is unbiased.

### Three Levels of CV (what runs when)

| Level | What | Splits | Purpose |
|-------|------|--------|---------|
| **Inner CV** | 5-fold StratifiedKFold on outer train | 5 | HP tuning — find best parameters without seeing test data |
| **Outer CV** | 5-fold StratifiedGroupKFold by gene | 5 | Model evaluation — unbiased performance estimate |
| **Multi-seed** | Repeat all folds with different seeds | 5 seeds | Robustness — mean ± std across seeds, not just one lucky split |

**Total: 5 outer × 5 inner × 5 seeds = 125 CV splits evaluated per model**

For model comparison runs: 3 seeds × 5 outer × 5 inner × 30 trials = 2,250 Optuna trials per model.

### CV Modes

#### 1. Gene-Level CV (`cv_mode="subunit"`) — PRIMARY

```python
groups = make_subunit_group_key(df)  # hash of gene name
StratifiedGroupKFold(n_splits=5, groups=groups)
```

- All variants of, say, CHRNA7 go into ONE fold
- The model is tested on genes it has NEVER seen in training
- **This is the strictest generalization test** — corresponding to "can we predict
  effect direction for a newly discovered variant in a known gene?"
- Stratification ensures each fold has roughly similar GOF/LOF/NNE proportions

#### 2. Homology-Class Transfer — SECONDARY EXPERIMENT

| Experiment | Train On | Test On | Biological Question |
|-----------|----------|---------|-------------------|
| alpha → rest | CHRNA1-7, A9, A10 | CHRNB1-4, CHRND, CHRNE, CHRNG | Can α-subunit knowledge transfer to β/δ/ε/γ? |
| beta → rest | CHRNB1-4 | CHRNA1-7, A9, A10, CHRND, CHRNE, CHRNG | Can β-subunit knowledge transfer to α/δ/ε/γ? |
| special → rest | CHRND, CHRNE, CHRNG | CHRNA1-7, A9, A10, CHRNB1-4 | Can muscle-specific subunit knowledge transfer to neuronal? |

This directly tests the **cross-family generalization** hypothesis. If, say,
alpha-trained models perform well on beta subunits, it suggests the features
capture biophysical principles rather than gene-specific patterns.

#### 3. Species Transfer CV — THIRD EXPERIMENT

Same human test folds evaluated under 3 training conditions:

| Condition | Training Data | Hypothesis |
|-----------|--------------|------------|
| **human_only** | Human train only | Baseline: what can we learn from human data alone? |
| **mouse_only** | ALL mouse data | Cross-species: can rodent data alone predict human? |
| **mixed** | Human train + ALL mouse + ALL rat | Augmentation: does adding rodent data help? Expect mixed ≥ human_only |

Paired comparison (same test folds across conditions) enables **Wilcoxon signed-rank test**
for statistical significance. This is the agnology/cross-species knowledge transfer angle
for the PSB 2027 paper.

### Metrics

| Metric | Formula | Why Used |
|--------|---------|----------|
| **Macro F1** | Mean(per-class F1) | **Primary metric & Optuna objective.** Treats GOF, LOF, NNE equally. Punishes model that ignores minority class. |
| **MCC** | (TP×TN - FP×FN) / sqrt(...) | Most honest single-number summary for imbalanced multi-class. -1 to +1, 0 = random. |
| **Balanced Accuracy** | Mean(per-class recall) | Version of accuracy that accounts for class imbalance. |
| **Per-class F1** | F1 for each of LOF/NNE/GOF | Shows which effect directions the model handles well vs poorly. |
| **Confusion Matrix** | Raw counts | Reveals systematic errors (e.g., does model confuse NNE with LOF more than NNE with GOF?) |

---
## Running Experiments

### Step 1: Download PDBs (one-time setup)
```bash
# Install DSSP
conda install -c salilab dssp

# Download experimental PDBs (6UW8, 7EKT, 7EKO, 6CNJ, 6PV7)
python scripts/download_pdbs.py

# Download AlphaFold structures (CHRNA9, CHRNA10)
python scripts/download_alphafold.py
```

### Step 2: Verify everything
```bash
python -c "
from vep_nachr2.data.loader import load_mutation_data
from vep_nachr2.features.structural import get_pdb_availability
df = load_mutation_data()
print()
import pprint
pprint.pprint(get_pdb_availability())
"
```

### Step 3: Clear old cache (if you ran without PDBs before)
```bash
rm data/processed/feature_cache_*.pkl
# On Windows PowerShell:
# Remove-Item data/processed/feature_cache_*.pkl
```

### Step 4: Run experiments

#### Quick test (2-3 minutes)
```bash
python scripts/run_experiment.py --test
```
Runs RF with 3 folds, 10 Optuna trials. Verifies pipeline works with PDB features.

#### Full comparison (hours)
```bash
python scripts/run_experiment.py --full
```
4 core models (LR, SVM, RF, LightGBM), 5 folds, 50 trials. Primary results for thesis.

#### All models (longer — overnight)
```bash
python scripts/run_experiment.py --compare
```
All 10 models, 3 seeds, 30 trials. Best for model selection and comparison table.

#### Ablation study
```bash
python scripts/run_experiment.py --ablation --model random_forest
```
Drops each feature group one at a time, reports impact on macro F1.
Key question: do structural features improve performance?

#### Species transfer
```bash
python scripts/run_experiment.py --species-transfer --model lightgbm
```
Tests cross-species augmentation hypothesis.

### Using the CLI directly (more control)
```bash
# Single model with custom trials
python -m vep_nachr2.training.runner single --model random_forest --n-trials 20 --n-folds 3

# Compare specific models
python -m vep_nachr2.training.runner compare --models rf lgbm xgb --n-trials 30

# Ablation with any model
python -m vep_nachr2.training.runner ablation --model lightgbm

# Species transfer
python -m vep_nachr2.training.runner species-transfer --model random_forest

# Drop feature groups manually (quick test: no structural)
python -m vep_nachr2.training.runner single --model rf --n-trials 10 \
    --drop structural_core structural_nachr conformational
```

### Expected run times

| Experiment | Models | Folds | Trials | ~Time |
|-----------|--------|-------|--------|-------|
| `--test` | 1 (RF) | 3 | 10 | 2-5 min |
| `--full` | 4 | 5 | 50 | 2-4 hours |
| `--compare` | 10 | 5 | 30 | 6-12 hours |
| `--ablation` | 1 | 5 | 30 | 1-2 hours |
| `--species-transfer` | 1 | 5 | 30 | 1-2 hours |

Times are approximate for a modern laptop. LightGBM is fastest, CatBoost slowest.
Run `--compare` overnight.

---
## Verification Checklist

- [ ] `load_mutation_data()` returns 797 variants, 3 classes, 16 genes
- [ ] `load_all_reference_sequences('human')` returns all 16 FASTA sequences
- [ ] Feature extraction produces (797, 66) matrix with 0 NaNs
- [ ] Single RF fold produces macro F1 > 0.3 (baseline ~0.45)
- [ ] PDB files downloaded for 6UW8, 7EKT, 7EKO, 6CNJ, 6PV7
- [ ] DSSP files generated for all PDBs
- [ ] Full CV run produces macro F1 with plausible std
- [ ] Ablation confirms structural features contribute to performance
- [ ] Species transfer: mixed condition ≥ human_only (cross-species hypothesis)

---
## Key Design Decisions

1. **3-class prediction** (GOF/LOF/NNE) — not binary, not pathogenicity
2. **Substitutions only** — clean feature space, indels deferred
3. **Gene-level CV** — strictest generalization test, no leakage
4. **VEP-ENAC architecture clone** — mature patterns, battle-tested on ion channels
5. **nAChR-specific structural features** — TMD, ligand proximity, interface
6. **Per-model imbalance handling** — different strategies for different model types
7. **Pipeline architecture** — each feature extractor is independently testable and droppable
8. **Graceful PDB degradation** — runs without PDBs, fill values for missing structures

---
## Future Extensions (Architecture-Ready)

- **ESM-2 embeddings**: EmbeddingExtractor placeholder, add `facebook/esm` dependency
- **GNN on 3D structure**: Placeholder in models/, needs PyTorch Geometric
- **Conformational features for all subunits**: Currently α7 only (7EKT vs 7EKO)
- **Mouse/rat reference sequences**: For rodent structural features
- **AlphaFold structures**: CHRNA9, CHRNA10 need manual download
- **Indel/frameshift support**: After substitution model works well